# Week 7: Logistic Regression Demo

## Beginner Walkthrough: Binary Outcomes in Plain Language

### What you will learn
- When to use logistic regression (and why OLS is not a good fit for yes/no outcomes)
- How to read coefficients and odds ratios in simple terms
- How to judge model quality using confusion matrix and ROC/AUC

### How to use this notebook
- Run cells from top to bottom
- After each chart, read the short interpretation notes
- Focus on direction and size of effects before worrying about formulas

### Real-world examples of binary outcomes
- Loan default: Yes or No
- Churn: Stay or Leave
- Job offer: Offer or No offer (our example)

---

## Section 1: Setup and Imports

We need several Python libraries:
- **NumPy & Pandas**: Data manipulation and analysis
- **Matplotlib**: Creating visualizations
- **Statsmodels**: Statistical modeling (including logistic regression)
- **Scikit-learn**: Machine learning utilities (ROC curve, AUC)

*Note: If scikit-learn isn't available, we'll use a fallback implementation.*

In [ ]:
# Standard data science imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Statistical modeling
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Try to import sklearn for ROC/AUC; provide fallback if not available
try:
    from sklearn.metrics import roc_curve, roc_auc_score
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    
    # Fallback implementations
    def roc_curve(y_true, y_score):
        thresholds = np.sort(np.unique(y_score))[::-1]
        fpr, tpr = [], []
        n_pos = (y_true == 1).sum()
        n_neg = (y_true == 0).sum()
        for t in np.append(thresholds, 0):
            pred = (y_score >= t).astype(int)
            tp = ((pred == 1) & (y_true == 1)).sum()
            fp = ((pred == 1) & (y_true == 0)).sum()
            tpr.append(tp / n_pos if n_pos else 0)
            fpr.append(fp / n_neg if n_neg else 0)
        return np.array(fpr), np.array(tpr), np.append(thresholds, 0)

    def roc_auc_score(y_true, y_score):
        fpr, tpr, _ = roc_curve(y_true, y_score)
        return float(np.sum(np.diff(fpr) * (tpr[:-1] + tpr[1:]) / 2))

print("✓ Imports successful!")
print(f"  NumPy version: {np.__version__}")
print(f"  Pandas version: {pd.__version__}")
print(f"  Scikit-learn available: {SKLEARN_AVAILABLE}")

In [ ]:
# Plot style settings for consistent, beginner-friendly visuals
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 10

PALETTE = {
    'negative': '#c44e52',
    'positive': '#4c72b0',
    'neutral': '#6c757d',
    'accent': '#55a868'
}

print('Plot style configured for consistent visuals.')

---

## Section 2: Data Generation

We'll simulate data for a realistic business scenario: **predicting whether a job candidate receives an offer** based on three features:

| Feature | Description | Range |
|---------|-------------|-------|
| **Education** | Years of formal education | 10-20 years |
| **Experience** | Years of work experience | 0-30 years |
| **Age** | Candidate age | 22-65 years |

**The True Model (hidden from us in real life):**
```
logit(p) = -8 + 0.5×Education + 0.1×Experience - 0.05×Age
```

Where `logit(p) = log(p / (1-p))` is the log-odds of getting a job offer.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Sample size
n = 500

# Generate independent variables (features)
# X1: Years of education (10-20)
education = np.random.uniform(10, 20, n)

# X2: Work experience (0-30 years)
experience = np.random.uniform(0, 30, n)

# X3: Age (22-65)
age = np.random.uniform(22, 65, n)

# Generate binary outcome: "Got Job Offer"
# True model: logit(p) = -8 + 0.5*education + 0.1*experience - 0.05*age
linear_combination = -8 + 0.5*education + 0.1*experience - 0.05*age
probability = 1 / (1 + np.exp(-linear_combination))
job_offer = (np.random.random(n) < probability).astype(int)

# Create DataFrame
df = pd.DataFrame({
    'Education': education,
    'Experience': experience,
    'Age': age,
    'JobOffer': job_offer
})

# Display summary
print(f"Generated {n} observations")
print(f"\nOutcome distribution:")
print(f"  No Job Offer: {(job_offer == 0).sum()} ({(job_offer == 0).mean()*100:.1f}%)")
print(f"  Job Offer:    {(job_offer == 1).sum()} ({(job_offer == 1).mean()*100:.1f}%)")
print("\nFirst 10 rows:")
df.head(10).round(2)

In [ ]:
# Quick visual profile of the simulated dataset
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: class balance
counts = df['JobOffer'].value_counts().sort_index()
axes[0].bar(['No Offer (0)', 'Offer (1)'], counts.values, color=['#c44e52', '#4c72b0'])
axes[0].set_title('Outcome Balance', fontsize=12)
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=10)

# Right: feature distributions
axes[1].hist(df['Education'], bins=18, alpha=0.6, label='Education', color='#4c72b0')
axes[1].hist(df['Experience'], bins=18, alpha=0.5, label='Experience', color='#55a868')
axes[1].hist(df['Age'], bins=18, alpha=0.45, label='Age', color='#8172b3')
axes[1].set_title('Feature Distributions', fontsize=12)
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

### Interpretation Checkpoint: Data Overview

- Left chart: this is class balance.
- Most observations are No Offer, so the classes are somewhat imbalanced.
- Right chart: features cover broad ranges, which is good for learning relationships.
- At this stage, we are checking data shape and balance, not model quality yet.

---

## Section 3: Explore the Data First

Before modeling, we ask two simple questions:
1. What does the data look like?
2. Does OLS make sensible probability predictions for a yes/no target?

### Plain-language takeaway
- A probability must be between 0 and 1
- If a model predicts below 0 or above 1, it is not appropriate for probability prediction
- Logistic regression was designed to avoid this issue

In [ ]:
# Descriptive statistics
print("Descriptive Statistics:")
print("="*60)
df.describe().round(2)

In [ ]:
# Demonstrate the OLS problem
print("\nWhy OLS is Problematic for Binary Outcomes:")
print("="*60)

# Run OLS (for demonstration - NOT recommended for binary outcomes)
X_ols = sm.add_constant(df[['Education', 'Experience', 'Age']])
model_ols = sm.OLS(df['JobOffer'], X_ols).fit()

# Check for predictions outside [0,1]
predictions = model_ols.fittedvalues
outside_range = ((predictions < 0) | (predictions > 1)).sum()

print(f"   - Predictions outside valid [0,1] range: {outside_range} cases")
print(f"   - Minimum prediction: {predictions.min():.3f} (should be ≥ 0)")
print(f"   - Maximum prediction: {predictions.max():.3f} (should be ≤ 1)")
print("\n   OLS assumes continuous outcomes with normal errors.")
print("   Binary outcomes VIOLATE these assumptions!")

### Interpretation Checkpoint: Why OLS Fails Here

- Any bar left of 0 or right of 1 is an invalid probability.
- This confirms the key issue: OLS is not built for binary outcomes.
- Logistic regression fixes this by constraining predictions to valid probabilities.

In [ ]:
# Visual check: OLS fitted values can fall outside [0, 1]
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(predictions, bins=30, color='#dd8452', edgecolor='white', alpha=0.9)
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Valid lower bound (0)')
ax.axvline(1, color='red', linestyle='--', linewidth=2, label='Valid upper bound (1)')
ax.set_title('OLS Fitted Values for Binary Outcome', fontsize=12)
ax.set_xlabel('OLS fitted value')
ax.set_ylabel('Count')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

---

## Section 4: Fit the Logistic Regression Model

Logistic regression models the chance of an event (here: getting a job offer).

### In simple terms
- Step 1: Build a weighted score from the predictors
- Step 2: Convert that score into a probability between 0 and 1

### Model form

$$\text{logit}(p) = \ln\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3$$

Where:
- $p$ is the probability of a job offer
- Positive coefficients raise the probability
- Negative coefficients lower the probability

In [ ]:
# Fit the logistic regression model
print("Fitting Logistic Regression Model...")
print("="*60)

# Add constant (intercept) to the independent variables
X_logit = sm.add_constant(df[['Education', 'Experience', 'Age']])
y = df['JobOffer']

# Fit the model
model_logit = Logit(y, X_logit).fit(disp=0)

print("\nModel fitted successfully!\n")
print("="*60)


In [ ]:
# Display the regression results table
print("\nLogistic Regression Results:")
print("-"*60)

# Create a nicely formatted results table
results_df = pd.DataFrame({
    'Variable': model_logit.params.index,
    'Coefficient': model_logit.params.values,
    'Std Error': model_logit.bse.values,
    'z-value': model_logit.tvalues.values,
    'P>|z|': model_logit.pvalues.values
})

# Add significance stars
def significance_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    return ''

results_df['Signif.'] = results_df['P>|z|'].apply(significance_stars)

# Display the table
print(results_df.to_string(index=False, float_format='%.4f'))
print("\nSignificance codes: *** p<0.001, ** p<0.01, * p<0.05")
print("-"*60)
print(f"Pseudo R-squared: {model_logit.prsquared:.4f}")
print(f"Log-Likelihood:   {model_logit.llf:.2f}")
print(f"AIC:              {model_logit.aic:.2f}")
print(f"BIC:              {model_logit.bic:.2f}")

---

## Section 5: Interpreting Odds Ratios

Coefficients in logistic regression are in **log-odds** units, which are hard to interpret directly. We convert them to **odds ratios** by exponentiating.

### What is an Odds Ratio?

$$\text{Odds Ratio} = e^{\beta} = \exp(\beta)$$

**Interpretation:**
- **OR > 1**: A one-unit increase in X **multiplies** the odds of the outcome by OR (increases odds)
- **OR < 1**: A one-unit increase in X **multiplies** the odds of the outcome by OR (decreases odds)
- **OR = 1**: No effect

**Example:** If OR for Education = 1.50, then each additional year of education **multiplies** the odds of getting a job offer by 1.5 (a 50% increase in odds).

In [ ]:
# Calculate odds ratios
odds_ratios = np.exp(model_logit.params)

print("ODDS RATIOS: exp(coefficient)")
print("="*60)
print(f"{'Variable':<15} {'Odds Ratio':>12} {'% Change':>15} {'Interpretation'}")
print("-"*60)

for var in ['Education', 'Experience', 'Age']:
    or_val = odds_ratios[var]
    pct_change = (or_val - 1) * 100
    
    if or_val > 1:
        direction = "increases odds"
        pct_str = f"+{pct_change:.1f}%"
    else:
        direction = "decreases odds"
        pct_str = f"{pct_change:.1f}%"
    
    print(f"{var:<15} {or_val:>12.4f} {pct_str:>15} {direction}")

print("-"*60)
print("\nINTERPRETATION EXAMPLES:")
print(f"  • Education: Each additional year → {(odds_ratios['Education']-1)*100:.1f}% HIGHER odds of job offer")
print(f"  • Experience: Each additional year → {(odds_ratios['Experience']-1)*100:.1f}% HIGHER odds")
print(f"  • Age: Each additional year → {(odds_ratios['Age']-1)*100:.1f}% LOWER odds (slight decrease)")

In [ ]:
# Visualize odds ratios
fig, ax = plt.subplots(figsize=(10, 6))

vars_plot = ['Education', 'Experience', 'Age']
ors = [odds_ratios[v] for v in vars_plot]
colors = ['green' if o > 1 else 'red' for o in ors]

bars = ax.barh(vars_plot, ors, color=colors, alpha=0.7, edgecolor='black')
ax.axvline(x=1, color='black', linestyle='--', linewidth=2, label='No Effect (OR = 1)')

ax.set_xlabel('Odds Ratio', fontsize=12)
ax.set_title('Odds Ratios by Variable\n(Green = Increases Odds, Red = Decreases Odds)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')

# Add value labels on bars
for bar, or_val in zip(bars, ors):
    ax.text(or_val + 0.02, bar.get_y() + bar.get_height()/2, 
            f'{or_val:.3f}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---

## Section 6: Predictions & Confusion Matrix

Once we have a fitted model, we can:
1. **Predict probabilities** - P(Job Offer = 1) for each candidate
2. **Classify** - Convert probabilities to binary predictions using a threshold (typically 0.5)
3. **Evaluate** - Use a confusion matrix to assess performance

### The Confusion Matrix

|                | Predicted: No | Predicted: Yes |
|----------------|---------------|----------------|
| **Actual: No** | True Negative (TN) | False Positive (FP) |
| **Actual: Yes**| False Negative (FN) | True Positive (TP) |

**Key Metrics:**
- **Accuracy** = (TP + TN) / Total - Overall correct predictions
- **Sensitivity (Recall)** = TP / (TP + FN) - Of actual positives, how many did we catch?
- **Specificity** = TN / (TN + FP) - Of actual negatives, how many did we catch?
- **Precision** = TP / (TP + FP) - Of predicted positives, how many were correct?

### Interpretation Checkpoint: Predicted Probabilities

- Blue bars (actual offers) should be more concentrated to the right (higher probabilities).
- Red bars (actual non-offers) should be more concentrated to the left (lower probabilities).
- Overlap between colors shows where the model is uncertain.
- The dashed line is the decision threshold used to turn probability into Yes/No.

In [ ]:
# Get predicted probabilities from the logistic model
df['Pred_Prob'] = model_logit.predict(X_logit)

# Classify using 0.5 threshold (default)
df['Pred_Class'] = (df['Pred_Prob'] > 0.5).astype(int)

# Show first few predictions
print("Sample Predictions:")
print("="*60)
sample_df = df[['Education', 'Experience', 'Age', 'JobOffer', 'Pred_Prob', 'Pred_Class']].head(10)
sample_df.round(4)

In [ ]:
# Predicted probability spread by actual class
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df.loc[df['JobOffer'] == 0, 'Pred_Prob'], bins=25, alpha=0.65, label='Actual No Offer (0)', color='#c44e52')
ax.hist(df.loc[df['JobOffer'] == 1, 'Pred_Prob'], bins=25, alpha=0.65, label='Actual Offer (1)', color='#4c72b0')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.8, label='Threshold = 0.5')
ax.set_title('Predicted Probabilities by Actual Outcome', fontsize=12)
ax.set_xlabel('Predicted probability of offer')
ax.set_ylabel('Count')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Build confusion matrix manually
actual_no = df[df['JobOffer'] == 0]
actual_yes = df[df['JobOffer'] == 1]

TN = (actual_no['Pred_Class'] == 0).sum()  # True Negatives
FP = (actual_no['Pred_Class'] == 1).sum()  # False Positives
FN = (actual_yes['Pred_Class'] == 0).sum()  # False Negatives
TP = (actual_yes['Pred_Class'] == 1).sum()  # True Positives

# Calculate metrics
accuracy = (TP + TN) / (TP + TN + FP + FN)
sensitivity = TP / (TP + FN)  # True positive rate (Recall)
specificity = TN / (TN + FP)   # True negative rate
precision = TP / (TP + FP) if (TP + FP) > 0 else 0

print("Confusion Matrix:")
print("="*60)
print("                    Predicted")
print("                 No Offer  |  Offer")
print("-"*50)
print(f"Actual No Offer:    {TN:>4}   |  {FP:>4}")
print(f"Actual Offer:       {FN:>4}   |  {TP:>4}")
print("-"*50)
print(f"\nAccuracy:   {accuracy*100:.1f}%")
print(f"Sensitivity (Recall): {sensitivity*100:.1f}% - Of those who got offers, we predicted correctly")
print(f"Specificity: {specificity*100:.1f}% - Of those who didn't, we predicted correctly")
print(f"Precision:  {precision*100:.1f}% - Of those we predicted to get offers, actually got them")

In [ ]:
# Visualize the confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))

cm = np.array([[TN, FP], [FN, TP]])
im = ax.imshow(cm, cmap='Blues')

# Set ticks and labels
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['No Offer', 'Offer'])
ax.set_yticklabels(['No Offer', 'Offer'])
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix\n(Counts of Correct and Incorrect Predictions)', fontsize=14, fontweight='bold')

# Add text annotations
labels = [['True Negative\n(TN)', 'False Positive\n(FP)'], 
          ['False Negative\n(FN)', 'True Positive\n(TP)']]
for i in range(2):
    for j in range(2):
        text = ax.text(j, i, f"{labels[i][j]}\n{cm[i, j]}",
                     ha="center", va="center", fontsize=11, fontweight='bold',
                     color="white" if cm[i,j] > cm.max()/2 else "black")

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### Interpretation Checkpoint: Confusion Matrix

- Top-left and bottom-right are correct predictions.
- Top-right and bottom-left are mistakes.
- If false negatives are costly in your business case, focus on reducing the bottom-left cell.

---

## Section 7: ROC Curve and AUC (Model Ranking Quality)

The ROC curve shows how sensitivity and false positive rate change as the threshold moves.

### Beginner interpretation
- Better models push the ROC curve toward the top-left corner
- The diagonal line is random guessing
- AUC summarizes the curve in one number

AUC guide:
- 0.5: no better than random
- 0.7 to 0.8: usable
- 0.8 to 0.9: strong
- above 0.9: excellent

In [ ]:
# Calculate ROC curve and AUC
fpr, tpr, thresholds = roc_curve(df['JobOffer'], df['Pred_Prob'])
auc_score = roc_auc_score(df['JobOffer'], df['Pred_Prob'])

# Create ROC curve plot
fig, ax = plt.subplots(figsize=(10, 8))

# Plot ROC curve
ax.plot(fpr, tpr, 'b-', linewidth=2.5, label=f'ROC Curve (AUC = {auc_score:.3f})')

# Plot random guess line
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.7, label='Random Guess (AUC = 0.5)')

# Mark the 0.5 threshold point
idx_05 = np.argmin(np.abs(thresholds - 0.5))
ax.plot(fpr[idx_05], tpr[idx_05], 'ro', markersize=10, label=f'Threshold = 0.5')

# Formatting
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('ROC Curve: Model Discrimination Ability\n(Higher AUC = Better Classifier)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Add AUC interpretation text
if auc_score >= 0.9:
    quality = "Outstanding"
elif auc_score >= 0.8:
    quality = "Excellent"
elif auc_score >= 0.7:
    quality = "Acceptable"
else:
    quality = "Poor"

ax.text(0.6, 0.15, f'AUC = {auc_score:.3f}\n{quality}', 
        fontsize=12, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"\nAUC Score: {auc_score:.4f}")
print(f"Interpretation: {quality} classifier")

---

## Section 8: One-Page Visual Recap

This panel combines the most important ideas in one place so students can review quickly:
- Probability curve
- Odds ratios
- Confusion matrix
- Threshold tuning

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Logistic Regression - Week 7 Demo\nBinary Outcome: Job Offer', 
             fontsize=14, fontweight='bold')

# Plot 1: Sigmoid curve showing probability vs education
ax1 = axes[0, 0]
x_range = np.linspace(10, 20, 100)
x_pred = np.column_stack([np.ones(100), x_range, 
                          np.full(100, df['Experience'].mean()),
                          np.full(100, df['Age'].mean())])
y_pred = model_logit.predict(x_pred)
ax1.plot(x_range, y_pred, 'b-', linewidth=2, label='Logistic Curve')
ax1.scatter(df['Education'], df['JobOffer'], alpha=0.3, s=20, c='red', label='Actual Data')
ax1.set_xlabel('Years of Education', fontsize=11)
ax1.set_ylabel('Probability of Job Offer', fontsize=11)
ax1.set_title('1. Logistic Sigmoid Curve\n(Probability vs Education)', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

# Plot 2: Odds ratios bar chart
ax2 = axes[0, 1]
vars_plot = ['Education', 'Experience', 'Age']
ors = [odds_ratios[v] for v in vars_plot]
colors = ['green' if o > 1 else 'red' for o in ors]
bars = ax2.barh(vars_plot, ors, color=colors, alpha=0.7)
ax2.axvline(x=1, color='black', linestyle='--', linewidth=1)
ax2.set_xlabel('Odds Ratio', fontsize=11)
ax2.set_title('2. Odds Ratios\n(Green > 1 = increases odds, Red < 1 = decreases)', fontsize=12)
for i, (bar, or_val) in enumerate(zip(bars, ors)):
    ax2.text(or_val + 0.02, bar.get_y() + bar.get_height()/2, 
             f'{or_val:.3f}', va='center', fontsize=10)

# Plot 3: Confusion matrix heatmap
ax3 = axes[1, 0]
cm = np.array([[TN, FP], [FN, TP]])
im = ax3.imshow(cm, cmap='Blues')
ax3.set_xticks([0, 1])
ax3.set_yticks([0, 1])
ax3.set_xticklabels(['No Offer', 'Offer'])
ax3.set_yticklabels(['No Offer', 'Offer'])
ax3.set_xlabel('Predicted', fontsize=11)
ax3.set_ylabel('Actual', fontsize=11)
ax3.set_title('3. Confusion Matrix', fontsize=12)
for i in range(2):
    for j in range(2):
        ax3.text(j, i, str(cm[i, j]), ha='center', va='center', 
                fontsize=16, fontweight='bold', 
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=ax3, shrink=0.8)

# Plot 4: Accuracy by threshold
ax4 = axes[1, 1]
thresholds = np.linspace(0.1, 0.9, 50)
accuracies = []
for thresh in thresholds:
    pred = (df['Pred_Prob'] > thresh).astype(int)
    acc = (pred == df['JobOffer']).mean()
    accuracies.append(acc)

ax4.plot(thresholds, accuracies, 'b-', linewidth=2)
ax4.axvline(x=0.5, color='red', linestyle='--', label='Default (0.5)')
best_idx = np.argmax(accuracies)
best_thresh = thresholds[best_idx]
ax4.axvline(x=best_thresh, color='green', linestyle='--', label=f'Best ({best_thresh:.2f})')
ax4.set_xlabel('Classification Threshold', fontsize=11)
ax4.set_ylabel('Accuracy', fontsize=11)
ax4.set_title('4. Accuracy by Threshold\n(Optimal threshold may not be 0.5)', fontsize=12)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nOptimal threshold: {best_thresh:.3f} (Accuracy: {accuracies[best_idx]*100:.1f}%)")
print(f"Default threshold (0.5): Accuracy: {df['Pred_Class'].eq(df['JobOffer']).mean()*100:.1f}%")

---

## Section 9: ROC Review (Optional Reinforcement)

This optional section repeats ROC/AUC with more detail for students who want extra practice interpreting the curve.

What to look for:
- Distance above the diagonal
- Where threshold 0.5 lands
- How AUC translates into ranking quality

In [ ]:
# Calculate ROC curve
fpr, tpr, roc_thresholds = roc_curve(df['JobOffer'], df['Pred_Prob'])
auc_score = roc_auc_score(df['JobOffer'], df['Pred_Prob'])

# Create ROC curve plot
fig, ax = plt.subplots(figsize=(10, 8))

# Plot ROC curve
ax.plot(fpr, tpr, 'b-', linewidth=2.5, label=f'ROC Curve (AUC = {auc_score:.3f})')

# Plot random guess line
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.7, label='Random Guess (AUC = 0.5)')

# Mark the 0.5 threshold point
idx_05 = np.argmin(np.abs(roc_thresholds - 0.5))
ax.plot(fpr[idx_05], tpr[idx_05], 'ro', markersize=10, label=f'Threshold = 0.5')

# Formatting
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('ROC Curve: Model Discrimination Ability\n(Higher AUC = Better Classifier)', 
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Add AUC interpretation
if auc_score >= 0.9:
    quality = "Outstanding"
elif auc_score >= 0.8:
    quality = "Excellent"
elif auc_score >= 0.7:
    quality = "Acceptable"
else:
    quality = "Poor"

ax.text(0.6, 0.15, f'AUC = {auc_score:.3f}\n{quality}', 
        fontsize=12, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"\nAUC Score: {auc_score:.4f}")
print(f"Interpretation: {quality} classifier")
print(f"\nThe model has a {auc_score:.1%} probability of ranking a random")
print(f"job offer candidate higher than a random non-offer candidate.")

### Interpretation Checkpoint: ROC and AUC

- Curves closer to the top-left are better.
- AUC near 1 means strong separation; AUC near 0.5 means near-random ranking.
- Use ROC/AUC to compare models before choosing a final threshold.

---

## Section 8: Key Concepts - Log-Odds and the Sigmoid

Let's visualize how logistic regression transforms linear predictions into probabilities.

In [ ]:
# Create visualization of log-odds to probability transformation
fig, axes2 = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Logistic Regression Concepts', fontsize=14, fontweight='bold')

# Chart A: Log-odds to probability (sigmoid)
ax_sigmoid = axes2[0]
z_range = np.linspace(-6, 6, 200)
p_from_z = 1 / (1 + np.exp(-z_range))
ax_sigmoid.plot(z_range, p_from_z, 'b-', linewidth=2, label='P(Y=1) = 1/(1+e^{-z})')
ax_sigmoid.axvline(x=0, color='gray', linestyle='--', alpha=0.7)
ax_sigmoid.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax_sigmoid.annotate('log-odds = 0\np = 0.5', xy=(0, 0.5), xytext=(1.5, 0.35),
                   fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))
ax_sigmoid.set_xlabel('Linear predictor z (log-odds)', fontsize=11)
ax_sigmoid.set_ylabel('P(Y=1)', fontsize=11)
ax_sigmoid.set_title('Log-Odds to Probability\n(Sigmoid transformation)', fontsize=12)
ax_sigmoid.legend()
ax_sigmoid.grid(True, alpha=0.3)
ax_sigmoid.set_xlim(-6, 6)
ax_sigmoid.set_ylim(0, 1)

# Chart B: ROC curve
ax_roc = axes2[1]
ax_roc.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc_score:.3f})')
ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random guess')
ax_roc.set_xlabel('False Positive Rate', fontsize=11)
ax_roc.set_ylabel('True Positive Rate', fontsize=11)
ax_roc.set_title('ROC Curve\n(Model discrimination ability)', fontsize=12)
ax_roc.legend()
ax_roc.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Summary: What to Remember

### 1. Pick the right model
- Use logistic regression for binary outcomes (0/1, yes/no)
- OLS can produce impossible probabilities (<0 or >1)

### 2. Interpret direction first
- Positive coefficient: higher predictor value increases chance of outcome
- Negative coefficient: higher predictor value decreases chance of outcome

### 3. Use odds ratios for practical meaning
- $OR = \exp(\beta)$
- OR > 1 means higher odds
- OR < 1 means lower odds

### 4. Evaluate with multiple metrics
- Confusion matrix: where errors happen
- Accuracy: overall correctness
- Sensitivity and specificity: error trade-off
- AUC: overall ranking quality

### 5. Threshold is a policy choice
- 0.5 is default, not always best
- Pick threshold based on business cost of false positives vs false negatives

In [ ]:
# Final Summary Statistics
print("\n" + "="*60)
print("FINAL SUMMARY: What You Just Learned")
print("="*60)

print("\nMODEL RESULTS:")
print(f"   - Sample size: {n} candidates")
print(f"   - Job offer rate: {(job_offer == 1).mean()*100:.1f}%")

print("\nODDS RATIOS:")
for var in ['Education', 'Experience', 'Age']:
    or_val = odds_ratios[var]
    pct = (or_val - 1) * 100
    print(f"   - {var}: OR = {or_val:.3f} ({pct:+.1f}% odds change per unit)")

print("\nCLASSIFICATION PERFORMANCE (Threshold = 0.5):")
print(f"   - True Negatives (correctly predicted no offer):  {TN}")
print(f"   - False Positives (incorrectly predicted offer):  {FP}")
print(f"   - False Negatives (missed actual offers):         {FN}")
print(f"   - True Positives (correctly predicted offers):    {TP}")
print(f"   - Accuracy:  {accuracy*100:.1f}%")
print(f"   - Sensitivity (Recall): {sensitivity*100:.1f}%")
print(f"   - Specificity: {specificity*100:.1f}%")

print("\nROC/AUC:")
print(f"   - AUC = {auc_score:.4f} ({quality} classifier)")
print(f"   - Interpretation: {auc_score:.1%} probability that a random job offer candidate")
print("     is ranked higher than a random non-offer candidate")

print("\n" + "="*60)
print("LOGISTIC REGRESSION DEMO COMPLETE!")
print("="*60)

---

## Appendix: The Math Behind Logistic Regression

For students who want to understand the mechanics:

### Step 1: Linear Predictor (Log-Odds)
$$z = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3$$

This is like OLS - a weighted sum of predictors. But $z$ can be any real number ($-\infty$ to $+\infty$).

### Step 2: Sigmoid Transformation
$$p = \frac{1}{1 + e^{-z}}$$

The sigmoid function "squashes" any real number into the range (0, 1), giving us a valid probability.

### Step 3: Maximum Likelihood Estimation
Unlike OLS which minimizes squared errors, logistic regression uses **Maximum Likelihood Estimation (MLE)** to find the coefficients that make the observed data most probable.

### Why This Matters
- OLS: Minimizes $\sum(y_i - \hat{y}_i)^2$ → can predict outside [0,1]
- Logit: Maximizes likelihood of observed outcomes → always predicts valid probabilities